# GCD Backbone → NASA GLOBE Fine-tuning

Domain-adaptive transfer: the DenseNet121 backbone pre-trained on GCD
(`best_cloudensenet_15_1.pt`) is loaded, its 7-class head is discarded,
and a fresh **TopBlock** for the 10 NASA GLOBE classes is trained with a
3-phase gradual-unfreezing schedule.

**Data pipeline** — identical to notebook 14: filtered NASA GLOBE images
via the ResNet18 cloud-filter CSV (confidence ≥ 0.99, up to 3 000 per class).

**Unfreezing schedule:**

| Phase | Layers unfrozen | Head LR | Backbone LR | Epochs | Condition |
|-------|----------------|---------|-------------|--------|-----------|
| 1 | TopBlock only | 1e-4 | — | 20 | always |
| 2 | TopBlock + denseblock4 + transition3 | 1e-4 | 1e-5 | 30 | always |
| 3 | + denseblock3 + transition2 | 1e-4 | 1e-5 / 5e-6 | 20 | Phase 2 improved >1 pp over Phase 1 |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

CSV_PATH    = Path("../resources/cloud-images/NASA_GLOBE_CD/cloud_filter_results_1.csv")
IMAGES_ROOT = Path("../resources/cloud-images/NASA_GLOBE_CD/downloaded_images")

THRESHOLDS = {
    "Ac": 0.99,
    "As": 0.99,
    "Cb": 0.99,
    "Cc": 0.99,
    "Ci": 0.99,
    "Cs": 0.99,
    "Cu": 0.99,
    "Ns": 0.99,
    "Sc": 0.99,
    "St": 0.99,
}

def index_labeled_images_filtered(csv_path, thresholds, images_root, max_per_class=3000):
    df = pd.read_csv(csv_path)
    df = df.rename(columns={"cloud_conf": "head2_conf", "head1_class": "head1_pred"})
    labeled_images = {}
    for folder, thresh in thresholds.items():
        clean = df[(df["folder"] == folder) & (df["head2_conf"] >= thresh)]
        clean = clean.sample(min(max_per_class, len(clean)), random_state=42)
        for _, row in clean.iterrows():
            labeled_images[row["filename"]] = {
                "label": folder,
                "path":  str(Path(images_root) / folder / row["filename"]),
            }
    return labeled_images

labeled_images = index_labeled_images_filtered(CSV_PATH, THRESHOLDS, IMAGES_ROOT)

counts = Counter(v["label"] for v in labeled_images.values())
for folder in sorted(counts):
    print(f"  {folder:6s}  {counts[folder]:,}")
print(f"  {'TOTAL':6s}  {sum(counts.values()):,}")

  Ac      3,000
  As      3,000
  Cb      3,000
  Cc      3,000
  Ci      3,000
  Cs      3,000
  Cu      3,000
  Ns      3,000
  Sc      3,000
  St      3,000
  TOTAL   30,000


In [2]:
def extract_labels(labeled_images):
    paths, labels = [], []
    for image_name in labeled_images:
        paths.append(labeled_images[image_name]["path"])
        labels.append(labeled_images[image_name]["label"])
    return paths, np.array(labels)

paths, labels = extract_labels(labeled_images)

In [3]:
import torch

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"device: {device}")

device: mps


In [4]:
import torchvision.transforms.v2 as T
from torchvision.transforms.v2 import InterpolationMode

IMG_SIZE = (224, 224)
MAX_FRAC = 0.14

border_translation = T.RandomAffine(
    degrees=0,
    translate=(MAX_FRAC, MAX_FRAC),
    interpolation=InterpolationMode.BILINEAR,
    fill=0
)

# T.Lambda is fine locally with num_workers=0
wrap_translation = T.Lambda(lambda x: torch.roll(
    x,
    shifts=(
        int(torch.randint(-int(MAX_FRAC * x.shape[-2]), int(MAX_FRAC * x.shape[-2]) + 1, (1,)).item()),
        int(torch.randint(-int(MAX_FRAC * x.shape[-1]), int(MAX_FRAC * x.shape[-1]) + 1, (1,)).item()),
    ),
    dims=(-2, -1),
))

stacked = T.Compose([
    T.RandomChoice([wrap_translation, border_translation]),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.15),
    T.RandomAffine(
        degrees=20, scale=(0.90, 1.10),
        interpolation=InterpolationMode.BILINEAR, fill=0
    ),
    T.RandomResizedCrop(
        size=IMG_SIZE, scale=(0.80, 1.00), ratio=(0.90, 1.10),
        interpolation=InterpolationMode.BILINEAR
    ),
])

one_of = T.RandomChoice([
    wrap_translation,
    border_translation,
    T.RandomChoice([T.RandomHorizontalFlip(p=1.0), T.RandomVerticalFlip(p=1.0)]),
    T.RandomRotation(degrees=20, interpolation=InterpolationMode.BILINEAR, fill=0),
])

normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transforms = T.Compose([
    T.Resize(IMG_SIZE, interpolation=InterpolationMode.BILINEAR),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.RandomChoice([stacked, one_of]),
    normalize,
])

eval_transforms = T.Compose([
    T.Resize(IMG_SIZE, interpolation=InterpolationMode.BILINEAR),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    normalize,
])

In [5]:
from sklearn.preprocessing import LabelEncoder

def encode_labels(labels):
    ordinal_encoder = LabelEncoder()
    encoded_labels = ordinal_encoder.fit_transform(labels)
    return encoded_labels, ordinal_encoder.classes_

encoded_labels, class_names = encode_labels(labels)
n_classes = len(class_names)
print(f"Classes ({n_classes}): {class_names}")

Classes (10): ['Ac' 'As' 'Cb' 'Cc' 'Ci' 'Cs' 'Cu' 'Ns' 'Sc' 'St']


In [6]:
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedShuffleSplit
from PIL import Image

class MyImages(Dataset):
    def __init__(self, paths, encoded_labels, split="train", test_size=0.2, val_size=0.1,
                 random_state=42, transform=None, split_indices=None):
        if split not in {None, "train", "val", "test"}:
            raise ValueError(f"split must be one of {{'None','train','val','test'}}, got {split!r}")

        self.transform = transform
        paths = np.array(list(paths))
        encoded_labels = np.array(list(encoded_labels))

        if split_indices is None:
            sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
            trainval_idx, test_idx = next(sss1.split(paths, encoded_labels))

            val_within_trainval = val_size / (1.0 - test_size)
            sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_within_trainval, random_state=random_state)
            train_rel_idx, val_rel_idx = next(sss2.split(paths[trainval_idx], encoded_labels[trainval_idx]))

            split_indices = {
                "train": trainval_idx[train_rel_idx],
                "val":   trainval_idx[val_rel_idx],
                "test":  test_idx,
            }

        idx = split_indices[split]
        self.paths = paths[idx].tolist()
        self.encoded_labels = encoded_labels[idx].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.encoded_labels[idx]


paths_arr  = np.array(list(paths))
labels_arr = np.array(list(encoded_labels))

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
trainval_idx, test_idx = next(sss1.split(paths_arr, labels_arr))

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.1/0.8, random_state=42)
train_rel_idx, val_rel_idx = next(sss2.split(paths_arr[trainval_idx], labels_arr[trainval_idx]))

split_indices = {
    "train": trainval_idx[train_rel_idx],
    "val":   trainval_idx[val_rel_idx],
    "test":  test_idx,
}

train_set = MyImages(paths=paths, encoded_labels=encoded_labels,
                     split="train", transform=train_transforms,
                     split_indices=split_indices)
valid_set = MyImages(paths=paths, encoded_labels=encoded_labels,
                     split="val",   transform=eval_transforms,
                     split_indices=split_indices)
test_set  = MyImages(paths=paths, encoded_labels=encoded_labels,
                     split="test",  transform=eval_transforms,
                     split_indices=split_indices)

print(f"train: {len(train_set):,}  val: {len(valid_set):,}  test: {len(test_set):,}")

train: 21,000  val: 3,000  test: 6,000


In [7]:
from torch.utils.data import DataLoader, WeightedRandomSampler

train_labels_list = [train_set.encoded_labels[i] for i in range(len(train_set))]
class_counts  = np.bincount(train_labels_list)
class_weights = 1.0 / class_counts
sample_weights = torch.tensor([class_weights[l] for l in train_labels_list], dtype=torch.float)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# num_workers=0 required locally — T.Lambda in transforms cannot be pickled by worker processes
train_loader = DataLoader(train_set, batch_size=64, sampler=sampler, num_workers=0)
valid_loader = DataLoader(valid_set, batch_size=64, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=64, num_workers=0)

In [8]:
import torchvision
import torch.nn as nn

GCD_N_CLASSES = 7
GCD_CKPT_PATH = 'best_cloudensenet_colab_15_1.pt'


class TopBlock(nn.Module):
    """
    Custom classification head from CloudDenseNet (Li et al., Sensors 2023).

    Replaces DenseNet121's single linear classifier with a 2-layer MLP:
      BN → Dropout → Linear(1024, hidden) → ReLU → BN → Dropout → Linear(hidden, n_classes)

    Weights are initialised with LeCun uniform distribution, as specified in the paper.
    """
    def __init__(self, in_features: int, n_classes: int,
                 hidden_dim: int = 512, dropout: float = 0.5):
        super().__init__()
        self.block = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(p=dropout),
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(p=dropout / 2),
            nn.Linear(hidden_dim, n_classes),
        )
        self._lecun_init()

    def _lecun_init(self):
        """LeCun uniform initialisation for Linear layers (as per paper)."""
        for m in self.block:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='linear')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.block(x)


# Step 1 — build DenseNet121 and attach a 7-class head to match checkpoint shape
weights = torchvision.models.DenseNet121_Weights.IMAGENET1K_V1
model = torchvision.models.densenet121(weights=weights).to(device)
model.classifier = TopBlock(in_features=1024, n_classes=GCD_N_CLASSES).to(device)

# Step 2 — load GCD backbone weights (backbone + 7-class head)
model.load_state_dict(torch.load(GCD_CKPT_PATH, map_location=device, weights_only=True))
print('GCD checkpoint loaded')

# Step 3 — discard GCD head, replace with a fresh 10-class TopBlock for NASA GLOBE
for param in model.parameters():
    param.requires_grad = False

model.classifier = TopBlock(in_features=1024, n_classes=n_classes).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Head replaced: GCD {GCD_N_CLASSES}-class → NASA GLOBE {n_classes}-class')
print(f'Classes ({n_classes}): {class_names}')
print(f'Trainable params: {trainable:,} / {total:,}')

GCD checkpoint loaded
Head replaced: GCD 7-class → NASA GLOBE 10-class
Classes (10): ['Ac' 'As' 'Cb' 'Cc' 'Ci' 'Cs' 'Cu' 'Ns' 'Sc' 'St']
Trainable params: 533,002 / 7,486,858


In [9]:
import torchmetrics


def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            metric.update(model(X_batch), y_batch)
    return metric.compute()


accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=n_classes).to(device)

In [10]:
import torch.nn.functional as F


class FocalLoss(nn.Module):
    """
    Focal Loss (Lin et al., 2017).

    Down-weights easy (high-confidence) examples so the model focuses on
    hard, misclassified ones.  gamma=0 reduces to standard cross-entropy.
    """
    def __init__(self, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        p_t = torch.exp(-ce)
        loss = ((1.0 - p_t) ** self.gamma) * ce
        return loss.mean() if self.reduction == 'mean' else loss.sum()


focal_loss = FocalLoss(gamma=2.0).to(device)
print("FocalLoss ready (gamma=2.0)")

FocalLoss ready (gamma=2.0)


In [11]:
def train_phase(model, optimizer, loss_fn, metric,
                train_loader, valid_loader,
                n_epochs, patience, checkpoint_path, phase_label=''):
    """
    Run one phase of the gradual-unfreeze schedule.

    Keeps frozen BatchNorm layers in eval mode so their running statistics
    are not corrupted; only BN layers with requires_grad=True are trained.
    Best weights are saved to checkpoint_path and restored before returning.
    """
    history    = {'train_losses': [], 'train_metrics': [], 'valid_metrics': []}
    best_val   = 0.0
    no_improve = 0

    for epoch in range(n_epochs):
        model.eval()
        model.classifier.train()
        for module in model.modules():
            if isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
                if any(p.requires_grad for p in module.parameters()):
                    module.train()

        total_loss = 0.0
        metric.reset()

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss   = loss_fn(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            metric.update(y_pred, y_batch)

        train_loss = total_loss / len(train_loader)
        train_acc  = metric.compute().item()
        val_acc    = evaluate_tm(model, valid_loader, metric).item()

        history['train_losses'].append(train_loss)
        history['train_metrics'].append(train_acc)
        history['valid_metrics'].append(val_acc)

        star = ' *' if val_acc > best_val else ''
        print(f'[{phase_label}] Epoch {epoch+1}/{n_epochs} | '
              f'loss: {train_loss:.4f} | '
              f'train: {train_acc:.4f} | '
              f'val: {val_acc:.4f}{star}')

        if val_acc > best_val:
            best_val   = val_acc
            no_improve = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stop at epoch {epoch+1} '
                      f'(best val: {best_val:.4f})')
                break

    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    return history, best_val

In [ ]:
all_history  = []
CKPT1 = 'best_nasa_phase1.pt'
CKPT2 = 'best_nasa_phase2.pt'
CKPT3 = 'best_nasa_phase3.pt'

# ── Phase 1 — Head only, lr=1e-4, 20 epochs ─────────────────────────────────
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

opt = torch.optim.AdamW(model.classifier.parameters(), lr=1e-4, weight_decay=1e-4)
h1, phase1_best_val = train_phase(model, opt, focal_loss, accuracy,
                                   train_loader, valid_loader,
                                   n_epochs=20, patience=10,
                                   checkpoint_path=CKPT1,
                                   phase_label='Phase 1 (head, 1e-4)')
all_history.append(h1)
print(f'Phase 1 best val: {phase1_best_val:.4f}')

# ── Phase 2 — Head + denseblock4 + transition3, 30 epochs ───────────────────
# Restore Phase 1 best before starting
model.load_state_dict(torch.load(CKPT1, weights_only=True))

for name, param in model.named_parameters():
    param.requires_grad = (
        name.startswith('classifier') or
        name.startswith('features.denseblock4') or
        name.startswith('features.transition3')
    )

head_params   = list(model.classifier.parameters())
block4_params = [p for n, p in model.named_parameters()
                 if p.requires_grad and not n.startswith('classifier')]

opt = torch.optim.AdamW([
    {'params': head_params,   'lr': 1e-4},
    {'params': block4_params, 'lr': 1e-5},
], weight_decay=1e-4)
h2, phase2_best_val = train_phase(model, opt, focal_loss, accuracy,
                                   train_loader, valid_loader,
                                   n_epochs=30, patience=10,
                                   checkpoint_path=CKPT2,
                                   phase_label='Phase 2 (block4+tr3, 1e-5)')
all_history.append(h2)
print(f'Phase 2 best val: {phase2_best_val:.4f}')

# ── Phase 3 — conditional: also unfreeze denseblock3 + transition2 ───────────
# Only runs if Phase 2 improved more than 1pp over Phase 1 best val
FINAL_CKPT  = CKPT2
improvement = phase2_best_val - phase1_best_val

if improvement > 0.01:
    print(f'Phase 2 gain {improvement:.4f} > 1pp — running Phase 3')

    # Restore Phase 2 best before starting
    model.load_state_dict(torch.load(CKPT2, weights_only=True))

    for name, param in model.named_parameters():
        param.requires_grad = (
            name.startswith('classifier') or
            name.startswith('features.denseblock4') or
            name.startswith('features.transition3') or
            name.startswith('features.denseblock3') or
            name.startswith('features.transition2')
        )

    head_params = list(model.classifier.parameters())
    block4_tr3  = [p for n, p in model.named_parameters()
                   if p.requires_grad and not n.startswith('classifier') and
                   (n.startswith('features.denseblock4') or n.startswith('features.transition3'))]
    block3_tr2  = [p for n, p in model.named_parameters()
                   if p.requires_grad and not n.startswith('classifier') and
                   (n.startswith('features.denseblock3') or n.startswith('features.transition2'))]

    opt = torch.optim.AdamW([
        {'params': head_params, 'lr': 1e-4},
        {'params': block4_tr3,  'lr': 1e-5},
        {'params': block3_tr2,  'lr': 5e-6},
    ], weight_decay=1e-4)
    h3, phase3_best_val = train_phase(model, opt, focal_loss, accuracy,
                                       train_loader, valid_loader,
                                       n_epochs=20, patience=8,
                                       checkpoint_path=CKPT3,
                                       phase_label='Phase 3 (block3+tr2, 5e-6)')
    all_history.append(h3)
    FINAL_CKPT = CKPT3
    print(f'Phase 3 best val: {phase3_best_val:.4f}')
else:
    print(f'Phase 2 gain {improvement:.4f} ≤ 1pp — skipping Phase 3')

print(f'\nFinal checkpoint: {FINAL_CKPT}')
model.load_state_dict(torch.load(FINAL_CKPT, weights_only=True))

In [ ]:
test_acc = evaluate_tm(model, test_loader, accuracy)
print(f'Final test accuracy: {test_acc:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = ['tab:blue', 'tab:green', 'tab:red']
phase_labels = [
    'Phase 1 (head only, 1e-4)',
    'Phase 2 (block4+tr3, 1e-5)',
    'Phase 3 (block3+tr2, 5e-6)',
]

offset = 0
for h, label, color in zip(all_history, phase_labels, colors):
    xs = list(range(offset, offset + len(h['train_losses'])))
    axes[0].plot(xs, h['train_losses'], color=color, label=label)
    axes[1].plot(xs, h['train_metrics'], color=color, linestyle='--', alpha=0.5)
    axes[1].plot(xs, h['valid_metrics'],  color=color, label=label)
    for ax in axes:
        ax.axvline(x=offset, color='grey', linewidth=0.5, linestyle=':')
    offset += len(h['train_losses'])

axes[0].set_title('Focal Loss — GCD backbone → NASA GLOBE')
axes[0].set_xlabel('Epoch (cumulative)')
axes[0].set_ylabel('Focal Loss')
axes[0].legend(fontsize=8)

axes[1].set_title('Accuracy — dashed=train, solid=val')
axes[1].set_xlabel('Epoch (cumulative)')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()